In [1]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from mne.decoding import CSP
import mne
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
import numpy as np
from sklearn.model_selection import LeaveOneGroupOut
from pickle import dump, load 
import sys
import os
import mlflow
import mlflow.sklearn

sys.path.append(os.path.abspath(os.path.join('..')))

from train.train_BCICIV import train_BCICIV
from data_load.load_data_BCICIV import load_all_subjects

mne.set_log_level('WARNING')

In [2]:
print(f"MNE version: {mne.__version__}")
import sklearn
print(f"Sklearn version: {sklearn.__version__}")

MNE version: 1.11.0
Sklearn version: 1.4.2


## LOAD TRAIN DATA FOR BCICIV DATASET

In [3]:
data_path = '../datasets/BCICIV_2a_gdf'

data = load_all_subjects(data_path)

/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A01T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A02T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

Loaded file: ../datasets/BCICIV_2a_gdf/A07T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A08T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A09T.gdf


## CSP + LDA MODEL

In [4]:
mlflow.set_experiment("BCI_Motor_Imagery_Classification_Experiment")

<Experiment: artifact_location='/Volumes/MyPassport/EEGNet-Motor-Imagery-BCI/notebooks/mlruns/1', creation_time=1771844576474, experiment_id='1', last_update_time=1771844576474, lifecycle_stage='active', name='BCI_Motor_Imagery_Classification_Experiment', tags={}, workspace='default'>

In [5]:
pipe1 = Pipeline([
    ('csp', CSP(n_components=4, log=True, norm_trace=False)),
    ('clf', LinearDiscriminantAnalysis())
])

param_grid1 = {
    # 'csp__n_components': [2, 4, 6],
    # 'csp__reg': [None, 'ledoit_wolf', 'oas'],
    # 'clf__solver': ['svd', 'lsqr', 'eigen']
}

In [6]:
loso = LeaveOneGroupOut()

with mlflow.start_run(run_name="LDA_with_CSP"):

    model, params, best_score = train_BCICIV(data, pipe1, loso, param_grid1)

    mlflow.log_params(params)
    mlflow.log_metric("F1_mean", best_score)

    mlflow.sklearn.log_model(model, "model_bci")

    print(f"MLflow register completed: {best_score}")

ValueError: 
All the 9 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
9 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/anaconda3/envs/py311ml/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/opt/anaconda3/envs/py311ml/lib/python3.11/site-packages/sklearn/base.py", line 1474, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/py311ml/lib/python3.11/site-packages/sklearn/pipeline.py", line 471, in fit
    Xt = self._fit(X, y, routed_params)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/py311ml/lib/python3.11/site-packages/sklearn/pip

## CSP + SVM MODEL

In [ ]:
pipe2 = Pipeline([
    ('csp', CSP(n_components=4, log=True, norm_trace=False)),
    ('clf', SVC(C=1.0, kernel='rbf'))
])

param_grid2 = {
    # 'csp__n_components': [2, 4, 6],
    # 'csp__reg': [None, 'ledoit_wolf', 'oas'],
    # 'clf__C': [0.1, 1, 10],
    # 'clf__gamma': ['scale', 'auto']
}

In [ ]:
loso = LeaveOneGroupOut()

with mlflow.start_run(run_name="SVM_with_CSP"):

    model, params, best_score = train_BCICIV(data, pipe2, loso, param_grid2)

    mlflow.log_params(params)
    mlflow.log_metric("F1_mean", best_score)

    mlflow.sklearn.log_model(model, "model_bci")

    print(f"MLflow register completed: {best_score}")

## CHANNEL AGGREGATION

In [ ]:
data_agg = data.copy()
data_agg['X'] = np.mean(data_agg['X'], axis=1, keepdims=True)

## CSP + LDA MODEL

In [ ]:
mlflow.set_experiment("BCI_Motor_Imagery_Classification_Experiment")

2026/02/23 09:43:30 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/02/23 09:43:30 INFO mlflow.store.db.utils: Updating database tables


<Experiment: artifact_location='file:///tmp/mlflow_artifacts', creation_time=1771836211025, experiment_id='1', last_update_time=1771836211025, lifecycle_stage='active', name='BCI_Motor_Imagery_Classification_Experiment', tags={}, workspace='default'>

In [ ]:
pipe1 = Pipeline([
    ('csp', CSP(n_components=4, log=True, norm_trace=False)),
    ('clf', LinearDiscriminantAnalysis())
])

param_grid1 = {
    # 'csp__n_components': [2, 4, 6],
    # 'csp__reg': [None, 'ledoit_wolf', 'oas'],
    # 'clf__solver': ['svd', 'lsqr', 'eigen']
}

In [ ]:
loso = LeaveOneGroupOut()

with mlflow.start_run(run_name="LDA_with_CSP_data_agg"):

    model, params, best_score = train_BCICIV(data_agg, pipe1, loso, param_grid1)

    mlflow.log_params(params)
    mlflow.log_metric("F1_mean", best_score)

    mlflow.sklearn.log_model(model, "model_bci")

    print(f"MLflow register completed: {best_score}")

## CSP + SVM MODEL

In [ ]:
pipe2 = Pipeline([
    ('csp', CSP(n_components=4, log=True, norm_trace=False)),
    ('clf', SVC(C=1.0, kernel='rbf'))
])

param_grid2 = {
    # 'csp__n_components': [2, 4, 6],
    # 'csp__reg': [None, 'ledoit_wolf', 'oas'],
    # 'clf__C': [0.1, 1, 10],
    # 'clf__gamma': ['scale', 'auto']
}

In [ ]:
loso = LeaveOneGroupOut()

with mlflow.start_run(run_name="SVM_with_CSP_data_agg"):

    model, params, best_score = train_BCICIV(data_agg, pipe2, loso, param_grid2)

    mlflow.log_params(params)
    mlflow.log_metric("F1_mean", best_score)

    mlflow.sklearn.log_model(model, "model_bci")

    print(f"MLflow register completed: {best_score}")